In [0]:
from pyspark.sql import functions as F

In [0]:
sql_server_name = "quant-cloud-server"
database_name = "my-database"
port = 1433

jdbc_url = f"""jdbc:sqlserver://{sql_server_name}.database.windows.net:{port};database={database_name};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"""

user = dbutils.secrets.get(scope="sc-rainbow-batch-04", key="sql-server-user")
password = dbutils.secrets.get(scope="sc-rainbow-batch-04", key="sql-server-password")

In [0]:
sql_query = "(select * from SalesLT.Product) temp"

df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", sql_query)
    .option("user", user)
    .option("password", password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

display(df)

In [0]:
info_query = """
(select TABLE_SCHEMA as table_schema, TABLE_NAME as table_name 
from information_schema.tables 
where TABLE_TYPE = 'BASE TABLE') t
"""

table_info_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", info_query)
    .option("user", user)
    .option("password", password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

display(table_info_df)

In [0]:
(table_info_df.collect()[0]).table_schema

In [0]:
dbutils.fs.mounts()

In [0]:
for table in table_info_df.collect():
    table_schema = table.table_schema
    table_name = table.table_name
    print(f"Loading {table_schema}.{table_name}")

    sql_query = f"(select * from {table_schema}.{table_name}) temp"

    try:
        df = (
            spark.read.format("jdbc")
            .option("url", jdbc_url)
            .option("dbtable", sql_query)
            .option("user", user)
            .option("password", password)
            .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
            .load()
        )

        target_path = f"/mnt/rainbow-container/SqlData/{table_schema}_{table_name}"
        df.write.format("csv").mode("overwrite").save(target_path)
        print(f"Successfully loaded {table_schema}.{table_name}")

    except Exception as e:
        print(f"Error loading {table_schema}.{table_name}: {e}")
